# Notebook 05: Model Comparison - Test Set Evaluation

Evaluation of all 7 trained models on the held-out PTB-XL test fold (fold 10).

| Model | Type | Params |
|---|---|---|
| FCN-Wang | baseline CNN | ~310 k (100%) |
| HeartBERT LoRA r=8 | RoBERTa + LoRA | ~0.7% of 125 M |
| HeartBERT DoRA r=8 | RoBERTa + DoRA | ~0.7% of 125 M |
| ECG-PT LoRA r=8 | GPT-2 + LoRA | ~0.3% of 117 M |
| ECG-PT DoRA r=8 | GPT-2 + DoRA | ~0.3% of 117 M |
| HuBERT-ECG LoRA r=8 | HuBERT + LoRA | ~1% of 93 M |
| HuBERT-ECG DoRA r=8 | HuBERT + DoRA | ~1% of 93 M |

Outputs saved to `results/05_model_comparison/`.

In [ ]:
import sys, os, warnings, json, gc
from pathlib import Path
sys.path.append('../')
os.environ['TRANSFORMERS_OFFLINE'] = '0'
warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', category=UserWarning)

import numpy as np
import matplotlib.pyplot as plt
import torch
import wfdb

from src.utils.config import CFG
from src.preprocessing.label_utils import load_all_labels, SUPERCLASSES

DATA_PATH    = CFG['data']['path']
RESULTS_PATH = CFG['paths']['results']
HUBERT_SIZE  = CFG['model']['hubert_size']
device       = 'cuda' if torch.cuda.is_available() else 'cpu'

OUT_DIR = Path(RESULTS_PATH) / '05_model_comparison'
FIG_DIR = OUT_DIR / 'figures'
OUT_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

EXPERIMENTS = [
    'fcn_wang_baseline',
    'heartbert_lora_r8',
    'heartbert_dora_r8',
    'ecgpt_lora_r8',
    'ecgpt_dora_r8',
    'hubert_ecg_lora_r8',
    'hubert_ecg_dora_r8',
]

MODEL_COLORS = {
    'fcn_wang_baseline':  '#7f7f7f',
    'heartbert_lora_r8':  '#1f77b4',
    'heartbert_dora_r8':  '#1f77b4',
    'ecgpt_lora_r8':      '#ff7f0e',
    'ecgpt_dora_r8':      '#ff7f0e',
    'hubert_ecg_lora_r8': '#2ca02c',
    'hubert_ecg_dora_r8': '#2ca02c',
}

all_results = {}

print(f'Device : {device}')
if torch.cuda.is_available():
    print(f'GPU    : {torch.cuda.get_device_name(0)}')
print(f'Output : {OUT_DIR}')

## 2. Data

Test fold = fold 10 (2 157 records, never seen during training).

- `X_test_lead` / `y_test`: single Lead II numpy arrays for HeartBERT and ECG-PT.
- `test_ds_full`: full 12-lead streaming dataset for HuBERT-ECG.

In [ ]:
from src.preprocessing.dataset_full import ECGDatasetFull

Y = load_all_labels(
    DATA_PATH + 'ptbxl_database.csv',
    DATA_PATH + 'scp_statements.csv',
)
test_df = Y[Y.strat_fold == 10]

LEAD_IDX = 1  # Lead II

def _load_lead(df, data_path, lead_idx=LEAD_IDX):
    X, y = [], []
    for i, (_, row) in enumerate(df.iterrows()):
        sig, _ = wfdb.rdsamp(data_path + row['filename_lr'])
        X.append(sig[:, lead_idx].astype(np.float32))
        y.append(np.array(row['label_vec'], dtype=np.float32))
        if (i + 1) % 500 == 0:
            print(f'  {i+1}/{len(df)} records loaded...')
    return np.stack(X), np.stack(y)

print('Loading test set (single-lead)...')
X_test_lead, y_test = _load_lead(test_df, DATA_PATH)

test_ds_full = ECGDatasetFull(test_df, DATA_PATH)

assert X_test_lead.shape[1] == 1000, X_test_lead.shape
assert y_test.shape[1] == 5, y_test.shape

print(f'X_test_lead  : {X_test_lead.shape}')
print(f'y_test       : {y_test.shape}')
print(f'test_ds_full : {len(test_ds_full):,} records')

## 3. Evaluate models

FCN-Wang metrics are loaded from the JSON saved by notebook 03.
Each PEFT model is loaded, evaluated, then deleted to free GPU memory.

In [ ]:
fcn_metrics = json.loads(
    (Path(RESULTS_PATH) / '03_preprocessing_pipeline' / 'metrics.json').read_text()
)
fcn_prof = json.loads(
    (Path(RESULTS_PATH) / 'fcn_wang_baseline' / 'profiling.json').read_text()
)

all_results['fcn_wang_baseline'] = {
    'auc_macro':          fcn_metrics['auc_macro'],
    'fmax':               fcn_metrics['fmax'],
    'auprc_macro':        fcn_metrics['auprc_macro'],
    'per_class_auc':      fcn_metrics['per_class_auc'],
    'auc_ci_95':          fcn_metrics.get('auc_ci_95', 'N/A'),
    'trainable_params':   fcn_prof['trainable_params'],
    'checkpoint_size_mb': fcn_prof['checkpoint_size_mb'],
}

assert abs(all_results['fcn_wang_baseline']['auc_macro'] - 0.9225) < 0.005, \
    f"Unexpected FCN-Wang AUC: {all_results['fcn_wang_baseline']['auc_macro']}"

print(f"FCN-Wang baseline: AUC {all_results['fcn_wang_baseline']['auc_macro']:.4f}")

In [ ]:
from src.models.heartbert import HeartBERTClassifier
from src.evaluation.metrics import full_eval

EXP  = 'heartbert_lora_r8'
prof = json.loads((Path(RESULTS_PATH) / EXP / 'profiling.json').read_text())

model = HeartBERTClassifier(num_labels=5)
model.load()
model.load_adapter(RESULTS_PATH + EXP + '/best_adapter')

logits  = model.predict_logits(X_test_lead)
results = full_eval(logits, y_test, run_bootstrap=True, n_bootstrap=1000)

all_results[EXP] = {
    **results,
    'trainable_params':   prof['trainable_params'],
    'checkpoint_size_mb': prof['checkpoint_size_mb'],
}
print(f"{EXP}: AUC {results['auc_macro']:.4f}")

del model
gc.collect()
torch.cuda.empty_cache()

In [ ]:
from src.models.heartbert import HeartBERTClassifier
from src.evaluation.metrics import full_eval

EXP  = 'heartbert_dora_r8'
prof = json.loads((Path(RESULTS_PATH) / EXP / 'profiling.json').read_text())

model = HeartBERTClassifier(num_labels=5)
model.load()
model.load_adapter(RESULTS_PATH + EXP + '/best_adapter')

logits  = model.predict_logits(X_test_lead)
results = full_eval(logits, y_test, run_bootstrap=True, n_bootstrap=1000)

all_results[EXP] = {
    **results,
    'trainable_params':   prof['trainable_params'],
    'checkpoint_size_mb': prof['checkpoint_size_mb'],
}
print(f"{EXP}: AUC {results['auc_macro']:.4f}")

del model
gc.collect()
torch.cuda.empty_cache()

In [ ]:
from src.models.ecgpt import ECGPTClassifier
from src.evaluation.metrics import full_eval

EXP  = 'ecgpt_lora_r8'
prof = json.loads((Path(RESULTS_PATH) / EXP / 'profiling.json').read_text())

model = ECGPTClassifier(num_labels=5)
model.load()
model.load_adapter(RESULTS_PATH + EXP + '/best_adapter')

logits  = model.predict_logits(X_test_lead)
results = full_eval(logits, y_test, run_bootstrap=True, n_bootstrap=1000)

all_results[EXP] = {
    **results,
    'trainable_params':   prof['trainable_params'],
    'checkpoint_size_mb': prof['checkpoint_size_mb'],
}
print(f"{EXP}: AUC {results['auc_macro']:.4f}")

del model
gc.collect()
torch.cuda.empty_cache()

In [ ]:
from src.models.ecgpt import ECGPTClassifier
from src.evaluation.metrics import full_eval

EXP  = 'ecgpt_dora_r8'
prof = json.loads((Path(RESULTS_PATH) / EXP / 'profiling.json').read_text())

model = ECGPTClassifier(num_labels=5)
model.load()
model.load_adapter(RESULTS_PATH + EXP + '/best_adapter')

logits  = model.predict_logits(X_test_lead)
results = full_eval(logits, y_test, run_bootstrap=True, n_bootstrap=1000)

all_results[EXP] = {
    **results,
    'trainable_params':   prof['trainable_params'],
    'checkpoint_size_mb': prof['checkpoint_size_mb'],
}
print(f"{EXP}: AUC {results['auc_macro']:.4f}")

del model
gc.collect()
torch.cuda.empty_cache()

In [ ]:
from src.models.hubert_ecg import HuBERTECGClassifier
from src.evaluation.metrics import full_eval
from torch.utils.data import DataLoader

EXP  = 'hubert_ecg_lora_r8'
prof = json.loads((Path(RESULTS_PATH) / EXP / 'profiling.json').read_text())

model = HuBERTECGClassifier(size=HUBERT_SIZE, num_labels=5)
model.load()
model.load_adapter(RESULTS_PATH + EXP + '/best_adapter')
model.to(device)
model.eval()

test_loader = DataLoader(
    test_ds_full,
    batch_size  = CFG['training']['batch_size_full'],
    shuffle     = False,
    num_workers = CFG['training']['num_workers'],
)

logits_list, labels_list = [], []
with torch.no_grad():
    for x, y in test_loader:
        logits_list.append(model(x.to(device)).cpu())
        labels_list.append(y)

logits  = torch.cat(logits_list).numpy()
labels  = torch.cat(labels_list).numpy()
results = full_eval(logits, labels, run_bootstrap=True, n_bootstrap=1000)

all_results[EXP] = {
    **results,
    'trainable_params':   prof['trainable_params'],
    'checkpoint_size_mb': prof['checkpoint_size_mb'],
}
print(f"{EXP}: AUC {results['auc_macro']:.4f}")

del model
gc.collect()
torch.cuda.empty_cache()

In [ ]:
from src.models.hubert_ecg import HuBERTECGClassifier
from src.evaluation.metrics import full_eval
from torch.utils.data import DataLoader

EXP  = 'hubert_ecg_dora_r8'
prof = json.loads((Path(RESULTS_PATH) / EXP / 'profiling.json').read_text())

model = HuBERTECGClassifier(size=HUBERT_SIZE, num_labels=5)
model.load()
model.load_adapter(RESULTS_PATH + EXP + '/best_adapter')
model.to(device)
model.eval()

test_loader = DataLoader(
    test_ds_full,
    batch_size  = CFG['training']['batch_size_full'],
    shuffle     = False,
    num_workers = CFG['training']['num_workers'],
)

logits_list, labels_list = [], []
with torch.no_grad():
    for x, y in test_loader:
        logits_list.append(model(x.to(device)).cpu())
        labels_list.append(y)

logits  = torch.cat(logits_list).numpy()
labels  = torch.cat(labels_list).numpy()
results = full_eval(logits, labels, run_bootstrap=True, n_bootstrap=1000)

all_results[EXP] = {
    **results,
    'trainable_params':   prof['trainable_params'],
    'checkpoint_size_mb': prof['checkpoint_size_mb'],
}
print(f"{EXP}: AUC {results['auc_macro']:.4f}")

del model
gc.collect()
torch.cuda.empty_cache()

## 4. Results table and plots

In [ ]:
print(f"{'Model':<28}  {'Test AUC':>9}  {'95% CI':<24}  {'Fmax':>7}  {'AUPRC':>7}  {'Params':>12}  {'Ckpt':>7}")
print('-' * 100)
for exp in EXPERIMENTS:
    if exp not in all_results:
        print(f"{exp:<28}  (not evaluated)")
        continue
    res    = all_results[exp]
    params = res.get('trainable_params', 0)
    ckpt   = res.get('checkpoint_size_mb', 0.0)
    ci     = res.get('auc_ci_95', 'N/A')
    print(
        f"{exp:<28}  {res['auc_macro']:>9.4f}  {ci:<24}  "
        f"{res['fmax']:>7.4f}  {res['auprc_macro']:>7.4f}  "
        f"{params:>12,}  {ckpt:>6.1f} MB"
    )

In [ ]:
exp_order = [e for e in EXPERIMENTS if e in all_results]
aucs      = [all_results[e]['auc_macro'] for e in exp_order]
colors    = [MODEL_COLORS[e] for e in exp_order]

fig_auc, ax = plt.subplots(figsize=(10, 5))
bars = ax.barh(exp_order, aucs, color=colors, alpha=0.85, edgecolor='white')

fcn_auc = all_results['fcn_wang_baseline']['auc_macro']
ax.axvline(fcn_auc, color='#444', linestyle='--', linewidth=1.5,
           label=f'FCN-Wang baseline {fcn_auc:.4f}')

for bar, auc in zip(bars, aucs):
    ax.text(auc + 0.001, bar.get_y() + bar.get_height() / 2,
            f'{auc:.4f}', va='center', fontsize=9)

ax.set_xlabel('Macro AUC (test fold 10)')
ax.set_title('Test Set Macro AUC: All 7 Models')
ax.set_xlim(0.7, 1.02)
ax.legend()
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
exp_order = [e for e in EXPERIMENTS if e in all_results]
data = np.array([
    [all_results[m]['per_class_auc'][sc] for sc in SUPERCLASSES]
    for m in exp_order
])

fig_heatmap, ax = plt.subplots(figsize=(10, 5))
im = ax.imshow(data, cmap='RdYlGn', vmin=0.7, vmax=1.0, aspect='auto')
plt.colorbar(im, ax=ax, label='AUC')

ax.set_xticks(range(len(SUPERCLASSES)))
ax.set_xticklabels(SUPERCLASSES)
ax.set_yticks(range(len(exp_order)))
ax.set_yticklabels(exp_order)

for i in range(len(exp_order)):
    for j in range(len(SUPERCLASSES)):
        ax.text(j, i, f'{data[i, j]:.3f}',
                ha='center', va='center', fontsize=9,
                color='black' if data[i, j] < 0.92 else 'white')

ax.set_title('Per-Class AUC: All Models on Test Set')
plt.tight_layout()
plt.show()

In [ ]:
families    = ['heartbert', 'ecgpt', 'hubert_ecg']
fam_labels  = ['HeartBERT', 'ECG-PT', 'HuBERT-ECG']
lora_aucs   = [all_results.get(f'{f}_lora_r8', {}).get('auc_macro', 0) for f in families]
dora_aucs   = [all_results.get(f'{f}_dora_r8', {}).get('auc_macro', 0) for f in families]

x     = np.arange(len(families))
width = 0.35

fig_lora_dora, ax = plt.subplots(figsize=(8, 5))
bars1 = ax.bar(x - width / 2, lora_aucs, width, label='LoRA r=8', color='#4c72b0', alpha=0.9)
bars2 = ax.bar(x + width / 2, dora_aucs, width, label='DoRA r=8', color='#c44e52', alpha=0.9)

for bar, v in zip(list(bars1) + list(bars2), lora_aucs + dora_aucs):
    if v > 0:
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.001,
                f'{v:.4f}', ha='center', fontsize=8)

ax.set_xticks(x)
ax.set_xticklabels(fam_labels)
ax.set_ylabel('Macro AUC')
ax.set_title('LoRA vs DoRA: Test Set AUC by Model Family')
ax.set_ylim(0.7, 1.05)
ax.legend()
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
MARKERS = {
    'fcn_wang_baseline':  ('o', 'FCN-Wang (all params)'),
    'heartbert_lora_r8':  ('s', 'HeartBERT LoRA'),
    'heartbert_dora_r8':  ('^', 'HeartBERT DoRA'),
    'ecgpt_lora_r8':      ('s', 'ECG-PT LoRA'),
    'ecgpt_dora_r8':      ('^', 'ECG-PT DoRA'),
    'hubert_ecg_lora_r8': ('s', 'HuBERT-ECG LoRA'),
    'hubert_ecg_dora_r8': ('^', 'HuBERT-ECG DoRA'),
}

fig_efficiency, ax = plt.subplots(figsize=(10, 6))

for exp, res in all_results.items():
    params = res.get('trainable_params', 1)
    auc    = res['auc_macro']
    marker, label = MARKERS.get(exp, ('o', exp))
    ax.scatter(params, auc, c=MODEL_COLORS.get(exp, '#333'),
               marker=marker, s=140, zorder=5)
    ax.annotate(label, (params, auc),
                textcoords='offset points', xytext=(8, 3), fontsize=8)

ax.set_xscale('log')
ax.set_xlabel('Trainable Parameters (log scale)')
ax.set_ylabel('Macro AUC (test)')
ax.set_title('Model Efficiency: Trainable Parameters vs Test AUC')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
def _serialisable(obj):
    """Recursively convert numpy scalars to Python primitives."""
    if isinstance(obj, dict):
        return {k: _serialisable(v) for k, v in obj.items()}
    if isinstance(obj, (np.float32, np.float64, np.floating)):
        return float(obj)
    if isinstance(obj, (np.int32, np.int64, np.integer)):
        return int(obj)
    return obj

with open(OUT_DIR / 'all_metrics.json', 'w') as f:
    json.dump(_serialisable(all_results), f, indent=2)

fig_auc.savefig(FIG_DIR / 'auc_comparison.png',     dpi=150, bbox_inches='tight')
fig_heatmap.savefig(FIG_DIR / 'per_class_heatmap.png', dpi=150, bbox_inches='tight')
fig_lora_dora.savefig(FIG_DIR / 'lora_vs_dora.png',    dpi=150, bbox_inches='tight')
fig_efficiency.savefig(FIG_DIR / 'efficiency_scatter.png', dpi=150, bbox_inches='tight')

print('Saved:')
for p in sorted(OUT_DIR.rglob('*')):
    if p.is_file():
        print(f'  {p.relative_to(Path(RESULTS_PATH))}  ({p.stat().st_size / 1024:.1f} KB)')